# Customer Support Ticket Forecasting

## Project Overview

Forecasts **daily customer support ticket volume** over a 14-day horizon using synthetic data spanning 2 years with weekly patterns, product-launch spikes, and gradual growth.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily ticket counts, predict the next 14 days. This drives agent scheduling, hiring decisions, and SLA management.

## Why This Project Matters

Under-staffed support teams lead to long response times and churn. Over-staffing wastes payroll. Accurate forecasts optimize the balance.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily support tickets
- Weekly pattern (higher Mon-Fri, lower weekends)
- Monday spike (backlog from weekend)
- Growth trend (expanding user base)
- Product launch spikes every ~90 days

| Column | Description |
|--------|-------------|
| `date` | Date |
| `tickets` | Daily support ticket count |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'tickets'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 400 + 0.3 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow == 0: weekly[i] = 100  # Monday
    elif dow <= 4: weekly[i] = 50  # Tue-Fri
    else: weekly[i] = -150  # Weekend

launches = np.zeros(N_POINTS)
for i in range(0, N_POINTS, 90):
    for j in range(min(5, N_POINTS - i)):
        launches[i + j] = 200

noise = np.random.normal(0, 30, N_POINTS)
tickets = base + weekly + launches + noise
tickets = np.maximum(tickets, 50).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'tickets': tickets})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,tickets
0,2022-01-01,465
1,2022-01-02,446
2,2022-01-03,720
3,2022-01-04,697
4,2022-01-05,644


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('tickets Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("tickets Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

tickets Statistics:
count    730.000000
mean     520.849315
std      126.827445
min      202.000000
25%      442.250000
50%      530.500000
75%      611.000000
max      939.000000
Name: tickets, dtype: float64

CV: 0.244
Skewness: -0.133


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:      201.8 | RMSE:      233.2 | MAPE: 27.43%
Seasonal Naive (7)             | MAE:      108.9 | RMSE:      138.6 | MAPE: 16.57%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                           Adjusted R-Squared  R-Squared        RMSE  Time Taken
Model                                                                           
KernelRidge                        364.648177 -26.972937  517.594112    0.056445
GaussianProcessRegressor            70.317695  -4.332130  225.980349    0.061512
MLPRegressor                        25.846265  -0.911251  135.294200    0.443392
DummyRegressor                      24.788384  -0.829876  132.382656    0.010402
QuantileRegressor                   21.928924  -0.609917  124.171540    0.061019
NuSVR                               14.819855  -0.063066  100.902139    0.033196
SVR                                 11.585530   0.185728   88.308990    0.033060
DecisionTreeRegressor               11.268223   0.210137   86.975366    0.016784
XGBRegressor                         8.759233   0.403136   75.606303    0.373028
GradientBoostingRegressor            6.962703   0.541331   66.278125    0.311650


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:       32.4 | RMSE:       43.3 | MAPE:  5.87%
FLAML Test (catboost)          | MAE:       77.3 | RMSE:      111.2 | MAPE: 10.41%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       77.1 | RMSE:      112.0 | MAPE: 10.41%
SF AutoETS                     | MAE:       73.9 | RMSE:      108.1 | MAPE: 10.01%
SF SeasonalNaive               | MAE:       72.6 | RMSE:      105.8 | MAPE:  9.86%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model        MAE       RMSE      MAPE
     FLAML (catboost)  32.364577  43.303519  5.870903
     SF SeasonalNaive  72.642860 105.812841  9.863376
           SF AutoETS  73.896797 108.082605 10.012565
         SF AutoARIMA  77.124931 111.987980 10.411996
FLAML Test (catboost)  77.253991 111.151574 10.409434
   Seasonal Naive (7) 108.857143 138.616119 16.574954
   Naive (Last Value) 201.785714 233.180647 27.432547

Top 3:
           model       MAE       RMSE      MAPE
FLAML (catboost) 32.364577  43.303519  5.870903
SF SeasonalNaive 72.642860 105.812841  9.863376
      SF AutoETS 73.896797 108.082605 10.012565


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 59.27, Std: 94.03


## Interpretation and Business Insight

- Monday spikes dominate the weekly pattern (weekend backlog)
- Product launches cause 3-5 day surges in ticket volume
- Gradual growth reflects expanding user base
- FLAML with lag features captures weekly cycle effectively
- 14-day forecasts align with agent scheduling cycles

## Limitations

1. Synthetic data — real tickets vary by category, severity, channel
2. No ticket type segmentation (billing vs. technical vs. general)
3. No self-service deflection modeling
4. Single aggregate — real teams forecast by queue/skill

## How to Improve This Project

1. Segment by ticket category for skill-based staffing
2. Add product release calendar as a feature
3. Model resolution time alongside volume
4. Add chatbot deflection rates
5. Use intraday forecasting for shift planning

## Production Considerations

- Daily automated retraining
- Integration with helpdesk platforms (Zendesk, Freshdesk)
- Real-time dashboard with forecast vs actual
- Alert on volume anomalies (outages, viral issues)

## Common Mistakes

1. Ignoring Monday spikes in feature engineering
2. Not accounting for product launches
3. Using shuffled CV on time series
4. Staffing to average instead of peak forecasts

## Mini Challenge / Exercises

1. Add a product launch feature and measure improvement
2. Forecast by ticket priority (P1, P2, P3)
3. Build an Erlang C staffing model on top of forecasts
4. Simulate SLA impact of under-forecasting by 20%

## Final Summary / Key Takeaways

1. Support ticket forecasting drives workforce planning
2. Monday spikes and product launches are key patterns
3. FLAML with lag features works well for stable patterns
4. 14-day horizons match typical scheduling cycles
5. Always compare against seasonal naive baseline

In [18]:
import json
metrics = {
    'project': 'Customer Support Ticket Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Customer Support Ticket Forecasting — Complete ===")

Metrics saved.

=== Customer Support Ticket Forecasting — Complete ===
